# 13. Coexistence of QoS flows on one NR sidelink Mode 2 channel

Can traffic with very different QoS requirements share one 20 MHz sidelink channel when every UE
selects its own resources? The notebook answers that from the full sweep (`results/qos/$QOS_SWEEP`,
default `qos_full`), for each resource allocation mechanism:

| Mechanism | What the UE does |
|---|---|
| **SB-SPS** | sensing-based semi-persistent scheduling: one sensing-based selection, kept for Cresel periods (TS 38.214 §8.1.4, TS 38.321 §5.22.1) |
| **DS** | dynamic scheduling: a new sensing-based selection for every packet, no reservation for the next period |
| **+ re-evaluation** | before signalling a selected resource, the UE checks it again (T3 = Tproc,1) and moves it if it is now taken |
| **+ re-evaluation + pre-emption** | also checks resources already announced, and leaves them when a higher priority UE claims them |

Pre-emption is only defined on top of re-evaluation in the sweep, so its contribution is measured
as *re-evaluation + pre-emption* against *re-evaluation*.

Sections:

1. the simulation: PHY, scenarios, background traffic, traffic flows and their colours
2. coexistence: every metric against the Tx-Rx distance, per scenario, mechanism and background level
3. why packets are lost: loss causes per scenario and background level
4. PDR against latency, with the requirement of each flow
5. what re-evaluation, pre-emption and both bring, overall then per flow
6. more: reliable range, requirement compliance, channel load, who gets pre-empted, platooning
   under impairments, SB-SPS against DS for CPMs, disconnection tails
7. what the data says, and its limits

In [ ]:
import os, sys, json, subprocess
sys.path.insert(0, os.path.abspath("../../tools"))
sys.path.insert(0, os.path.abspath("../../tools/qos"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

mr.apply_chart_style()
os.chdir(mr.ROOT)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

import glob
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import plots

s = plots.Sweep()
print(s.note())
SWEEP = os.path.join(mr.ROOT, "results", "qos", s.name)
flows, by_distance, causes, histograms = s.flows, s.by_distance, s.causes, s.histograms

RANGE_M = 300        # m, the range of the per-flow summaries (PDR, loss causes, contributions)
MIN_ATTEMPTS = 50    # a distance bin of a run needs this many attempts to be plotted
T95 = {2: 12.71, 3: 4.30, 4: 3.18, 5: 2.78, 6: 2.57, 7: 2.45, 8: 2.36, 9: 2.31, 10: 2.26}

def ci95(values):
    """Mean and 95 % half-width of a t interval over independent samples (the seeds)."""
    v = np.asarray(values, dtype=float); v = v[~np.isnan(v)]
    if len(v) < 2:
        return (v.mean() if len(v) else np.nan), np.nan
    return v.mean(), T95.get(len(v), 1.96) * v.std(ddof=1) / np.sqrt(len(v))

def logit(p): return np.log(p / (1 - p))
def expit(z): return 1 / (1 + np.exp(-z))

def proportion_ci(p, attempts):
    """Plain mean of a proportion over the seeds, and its 95 % interval computed on the logit scale,
    so the interval never leaves [0, 1]. A seed at 0 or 1 counts half an attempt off the bound."""
    p, attempts = np.asarray(p, dtype=float), np.asarray(attempts, dtype=float)
    floor = 0.5 / attempts
    m, h = ci95(logit(np.clip(p, floor, 1 - floor)))
    h = 0 if np.isnan(h) else h
    return p.mean(), expit(m - h), expit(m + h)

def ci95_by(frame, by, value):
    """Per group of `by`: mean m and 95 % half-width h over the rows (one row per seed)."""
    g = frame.groupby(by)[value].agg(["mean", "std", "count"])
    g["h"] = [T95.get(int(n), 1.96) * sd / np.sqrt(n) if n > 1 else np.nan for sd, n in zip(g["std"], g["count"])]
    return g.rename(columns={"mean": "m"})[["m", "h"]]

## 1. The simulation

### 1.1 Radio and protocol

All runs share one configuration: NR sidelink Mode 2, numerology 2 (60 kHz, 0.25 ms slots),
**20 MHz = 24 RBs in two subchannels of 12 RBs**, MCS 14, 23 dBm, sensing threshold −128 dBm,
blind retransmissions per flow (1 for a PER of 10⁻⁴, 2 for 10⁻⁵, within 32 slots and the PDB).
Each run simulates 15 s, the first 3 s are left out. A transmission *attempt* is one packet and one
receiver within 1000 m. A CPM replaced in the queue by a newer CPM before it was sent
(*superseded*, about 19 % of the CPMs) is no attempt: the newer CPM carries the fresher state, so
it counts neither as delivered nor as lost, in every metric of this notebook. Every scenario runs with 3 seeds; a seed is a different SUMO trace, and
all mechanisms of one seed replay the same trace, so the mechanisms are compared on identical
mobility (paired differences in section 5).

Two subchannels make the channel coarse: one subchannel carries roughly half of the 637 B slot
payload, so any packet above about 300 B, and every 400 B or 600 B reservation, takes the whole
slot.

In [ ]:
cfg = yaml.safe_load(open(os.path.join(SWEEP, "sweep.yaml")))
pd.DataFrame({"value": cfg["phy"]}).T

### 1.2 Scenarios and background traffic

The **background traffic** is the vehicles that are not part of the flows under study (platoons,
emergency vehicles, RSUs, vulnerable road users), and the CAMs, CPMs and advanced driving messages
they send. Its level is the vehicle density of the SUMO trace. The sweep has two levels per
scenario, named as in `make_traces.py`. A run without background traffic is not part of
this sweep, and the plots take any level present in the data, so a later sweep with `none`
adds a column.

In [ ]:
LEVEL_ORDER = ["none", "low", "medium", "high"]
SCENARIO_ORDER = ["highway_platoon", "urban_intersection", "highway_cpm"]
SCENARIO_LABEL = {"highway_platoon": "Highway platooning", "urban_intersection": "Urban intersections",
                  "highway_cpm": "Highway CPM contention"}
LEVEL_MARKER = {"none": "o", "low": "s", "medium": "D", "high": "^"}

TRACES = {}
for path in glob.glob(os.path.join(SWEEP, "traces", "*", "meta.json")):
    meta = json.load(open(path))
    roles = pd.read_csv(os.path.join(os.path.dirname(path), "roles.csv"))
    meta["roles"] = roles.role.value_counts().to_dict()
    TRACES.setdefault((meta["scenario"], meta["density_name"]), []).append(meta)

def levels(scenario):
    present = set(flows[flows.scenario == scenario].density)
    return [l for l in LEVEL_ORDER if l in present]

def level_label(scenario, level):
    value = TRACES[(scenario, level)][0]["density"]
    unit = "vehicles in the grid" if scenario == "urban_intersection" else "veh/km/lane"
    return "%s background (%g %s)" % (level, value, unit)

rows = []
for scenario in SCENARIO_ORDER:
    for level in levels(scenario):
        metas = TRACES[(scenario, level)]
        roles = pd.DataFrame([m["roles"] for m in metas]).fillna(0).mean()
        rows.append({"scenario": scenario, "background": level, "density": metas[0]["density"],
                     "nodes (mean over seeds)": round(np.mean([m["nodes"] for m in metas])),
                     **{r: int(round(n)) for r, n in roles.items()}})
pd.DataFrame(rows).fillna(0).set_index(["scenario", "background"])

| Scenario | Road | Flows under study | Background | Conditions / variants |
|---|---|---|---|---|
| `highway_platoon` | 3 + 3 lane highway, 3 truck platoons of 6 | platooning, crash mitigation, emergency trajectories | cars and trucks with CAM, CPM, advanced driving | baseline, jamming, loss bursts, extra stack latency |
| `urban_intersection` | 4 × 4 grid, 200 m blocks with buildings (TR 37.885 UMi, LOS / NLOS / NLOSv) | VAMs of cyclists and pedestrians, RSU CPMs, crash mitigation, emergency | cars with CAM, CPM, advanced driving | baseline |
| `highway_cpm` | 3 + 3 lane highway | CPMs of variable size | every vehicle sends CAMs and CPMs | CPMs on SB-SPS or on DS |

**Emergency vehicles exist only at high background.** `make_traces.py` turns each car into an
emergency vehicle with probability 2 %, drawing from one random sequence per seed that every
scenario and density of that seed reuses. The first draws under 2 % come at the 80th, 141st and
166th car of seeds 1, 2 and 3, and the low-background traces hold 54 to 78 cars, so none of them
has an emergency vehicle. At high background there are 5, 3 and 1 (highway platooning) and 3, 2
and 1 (urban) emergency vehicles per seed, and the single one of urban seed 3 sent 2 packets in
the measured window. The emergency trajectory results therefore rest on a handful of vehicles,
and their intervals are wide. In `highway_cpm` the emergency vehicles only send CAMs.

The `highway_cpm` variant `sps_median` (reserving for a 400 B CPM) gives the same results as
`sps_max` (600 B) to every digit: both reservations take the two subchannels, so no size
reselection ever happens. It is left out below.

### 1.3 Traffic flows and their colours

One colour, marker and line style per flow, the same in every figure. The colours are the
Okabe-Ito set with black replaced by wine, checked for colour-vision deficiencies: every pair stays
apart under protanopia, deuteranopia and tritanopia except cooperative perception and advanced
driving (ΔE 7.6 under deuteranopia), which therefore also differ in marker and line style.

In [ ]:
FLOW_STYLE = {   # colour, marker, line style
    "crash_mitigation":       ("#D55E00", "D", "-"),
    "emergency_trajectory":   ("#882255", "v", "--"),
    "platooning":             ("#0072B2", "o", "-"),
    "advanced_driving":       ("#009E73", "s", "-."),
    "cam":                    ("#E69F00", "^", "-"),
    "vam":                    ("#56B4E9", "P", "--"),
    "cooperative_perception": ("#CC79A7", "X", ":"),
}
FLOW_LABEL = {
    "crash_mitigation": "Crash mitigation (PQI 91)",
    "emergency_trajectory": "Emergency trajectory (PQI 89)",
    "platooning": "Platooning (PQI 21)",
    "advanced_driving": "Advanced driving (PQI 22)",
    "cam": "CAM (PQI 90)",
    "vam": "VAM (PQI 90)",
    "cooperative_perception": "Cooperative perception, CPM (PQI 23)",
}
FLOW_ORDER = list(FLOW_STYLE)
def color(name): return FLOW_STYLE[name][0]
def ordered(names): return [n for n in FLOW_ORDER if n in set(names)]

classes = yaml.safe_load(open(os.path.join(mr.ROOT, "scenarios", "qos", "classes.yaml")))["classes"]
def model_text(c):
    if c["model"] == "periodic":
        return "periodic, %d ms" % c["period_ms"]
    if c["model"] == "aperiodic":
        return "%d ms + Exp(mean %d ms)" % (c["period_ms"], c["exp_mean_ms"])
    return "events: %.2f /s per vehicle, burst of %d every %d ms" % (c["event_rate"], c["burst"], c["burst_interval_ms"])
present = flows.groupby("name").scenario.unique()
traffic = pd.DataFrame([{
    "flow": FLOW_LABEL[n], "PQI": c["pqi"], "resource type": "GBR" if c["gbr"] else "non-GBR",
    "SL priority": c["priority"], "PDB [ms]": c["pdb_ms"], "PER target": "%.0e" % c["per"],
    "generation": model_text(c),
    "size [B]": "/".join(str(x) for x in c["sizes"]) + ("" if len(c["sizes"]) == 1 else " (p " + "/".join(str(p) for p in c.get("probs", [])) + ")"),
    "scheduling": "SB-SPS, RRI %d ms" % c["rri_ms"] if c["scheduling"] == "sps" else "DS",
    "blind retx": c["retx"], "sent by": ", ".join(c["roles"]) + ("" if c.get("share", 1) == 1 else " (%d %%)" % (100 * c["share"])),
    "scenarios": ", ".join(present.get(n, [])),
} for n, c in classes.items() if n in FLOW_STYLE]).set_index("flow")
traffic

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.2))
for i, name in enumerate(FLOW_ORDER):
    col, mk, ls = FLOW_STYLE[name]
    y = len(FLOW_ORDER) - i
    ax.plot([0, 1], [y, y], color=col, ls=ls, lw=2.5, marker=mk, markersize=8, markevery=[1])
    ax.text(1.08, y, FLOW_LABEL[name], va="center", fontsize=9)
ax.set_xlim(-0.05, 3.2); ax.set_ylim(0.3, len(FLOW_ORDER) + 0.7); ax.axis("off")
ax.set_title("Flow colours, markers and line styles used in every figure")
plt.show()

### 1.4 What the scenarios look like

A snapshot of seed 1 of every scenario and background level at t = 3 s, the first instant that
counts in the metrics, drawn from the SUMO trace ns-3 replays. The short line behind each node is
its path over the previous second. Roads, sidewalks, junctions and buildings come from the SUMO
network and `buildings.csv`.

The background vehicles are grey. The nodes that carry the flows under study have the colour of
their main flow: platoon trucks in platooning blue, emergency vehicles in emergency-trajectory
wine, cyclists and pedestrians in VAM blue, RSUs in CPM pink. The legend gives every role with the
number of nodes on the map and the flows they send (a share in % when only part of them does).

On the highway the vertical axis is stretched (the road is 2 km long and 19 m wide), so distances
are only true along the road. Every node sits on one line: two nodes a few hundred metres apart on
either side of a platoon both reach it but cannot sense each other, the hidden-node geometry that
section 3 measures. The jammer of the `jamming` condition is drawn at its position, 25 m from the
road centre line; it only exists in that condition.

In [ ]:
import xml.etree.ElementTree as ET
from matplotlib.collections import PolyCollection

ROLE_STYLE = {   # colour, marker, size, label
    "car":        ("#b8b7b0", "o", 10, "car"),
    "truck":      ("#5d5c58", "s", 16, "truck"),
    "platoon":    (color("platooning"), "s", 22, "platoon truck"),
    "emergency":  (color("emergency_trajectory"), "*", 90, "emergency vehicle"),
    "cyclist":    (color("vam"), "D", 18, "cyclist"),
    "pedestrian": (color("vam"), "o", 12, "pedestrian"),
    "rsu":        (color("cooperative_perception"), "^", 110, "RSU (static)"),
}
SHORT_FLOW = {"crash_mitigation": "crash", "emergency_trajectory": "emergency", "platooning": "platooning",
              "advanced_driving": "adv. driving", "cam": "CAM", "vam": "VAM", "cooperative_perception": "CPM"}
REFERENCE_VARIANT = {"highway_platoon": "default", "urban_intersection": "default", "highway_cpm": "sps_max"}

def role_flows(scenario):
    """Role -> 'CAM, CPM 50 %, ...' from the flow file QOS ran with."""
    spec = pd.read_csv(os.path.join(SWEEP, "flows", "%s_baseline_%s.csv" % (scenario, REFERENCE_VARIANT[scenario])))
    out = {}
    for _, f in spec.iterrows():
        for role in f.roles.split("|"):
            text = SHORT_FLOW[f["name"]] + ("" if f.share >= 1 else " %d %%" % round(100 * f.share))
            out.setdefault(role, []).append(text)
    return {r: ", ".join(v) for r, v in out.items()}

def road_polygons(net_file):
    """Lanes, sidewalks and junctions of a SUMO network as polygons in metres."""
    root = ET.parse(net_file).getroot()
    lanes, walks, junctions = [], [], []
    for edge in root.iter("edge"):
        if edge.get("function") == "internal":
            continue
        for lane in edge.iter("lane"):
            pts = np.array([p.split(",") for p in lane.get("shape").split()], dtype=float)
            half = float(lane.get("width", 3.2)) / 2
            for a, b in zip(pts[:-1], pts[1:]):
                d = b - a
                n = np.array([-d[1], d[0]]) / np.hypot(*d) * half
                (walks if lane.get("allow") == "pedestrian" else lanes).append([a + n, b + n, b - n, a - n])
    for j in root.iter("junction"):
        if j.get("type") != "internal" and j.get("shape"):
            junctions.append(np.array([p.split(",") for p in j.get("shape").split()], dtype=float))
    return lanes, walks, junctions

def draw_scenario(ax, scenario, level, seed=1, t=3.0, tail=1.0):
    folder = os.path.join(SWEEP, "traces", "%s_%s_%d" % (scenario, level, seed))
    if not os.path.exists(os.path.join(folder, "trace.csv")):
        ax.text(0.5, 0.5, "trace.csv missing: copy the traces with tools/qos/remote/download.sh",
                transform=ax.transAxes, ha="center")
        return
    lanes, walks, junctions = road_polygons(glob.glob(os.path.join(folder, "sumo", "*.net.xml"))[0])
    ax.add_collection(PolyCollection(junctions, facecolors="#d8d7d1", edgecolors="none", zorder=0))
    ax.add_collection(PolyCollection(walks, facecolors="#ecebe6", edgecolors="none", zorder=0))
    ax.add_collection(PolyCollection(lanes, facecolors="#d8d7d1", edgecolors="white", linewidths=0.6, zorder=0))
    buildings = os.path.join(folder, "buildings.csv")
    if os.path.exists(buildings):
        polys = [np.array(c.split(), dtype=float).reshape(-1, 2) for c in pd.read_csv(buildings).corners]
        ax.add_collection(PolyCollection(polys, facecolors="#8a8985", edgecolors="none", alpha=0.55, zorder=0))
    trace = pd.read_csv(os.path.join(folder, "trace.csv"))
    roles = pd.read_csv(os.path.join(folder, "roles.csv"))
    trace = trace.merge(roles[["node_id", "role", "group"]], on="node_id")
    recent = trace[(trace.time >= t - tail - 1e-9) & (trace.time <= t + 1e-9)].sort_values("time")
    now = recent[np.isclose(recent.time, t)]
    # static = trace[(trace.role == "rsu") & ~trace.node_id.isin(now.node_id)].drop_duplicates("node_id")
    # now = pd.concat([now, static])   # RSUs are written once at the start and once at the end
    flows_of = role_flows(scenario)
    for role in ROLE_STYLE:
        col, mk, size, label = ROLE_STYLE[role]
        nodes = now[now.role == role]
        if nodes.empty:
            continue
        for _, path in recent[recent.role == role].groupby("node_id"):
            ax.plot(path.x, path.y, color=col, lw=0.8, alpha=0.5, zorder=2)
        ax.scatter(nodes.x, nodes.y, s=size, marker=mk, color=col, edgecolors="white" if role != "rsu" else "#1f1e1c",
                   linewidths=0.4 if role != "rsu" else 0.8, zorder=4 if role in ("platoon", "emergency", "rsu") else 3,
                   label="%s × %d: %s" % (label, len(nodes), flows_of.get(role, "no flow")))
    for g, members in now[now.role == "platoon"].groupby("group"):
        ax.annotate("platoon %d" % (g + 1), (members.x.mean(), members.y.mean()), xytext=(0, 12 if members.y.mean() > 0 else -16),
                    textcoords="offset points", ha="center", fontsize=8, color="#1f1e1c")
    ax.set_title(level_label(scenario, level), fontsize=10)

def highway_map(scenario, jammer=None):
    lvls = levels(scenario)
    fig, axes = plt.subplots(len(lvls), 1, figsize=(15, 2.9 * len(lvls)), squeeze=False)
    for ax, level in zip(axes[:, 0], lvls):
        draw_scenario(ax, scenario, level)
        ax.set_xlim(-10, 2010); ax.set_ylim(-32 if jammer else -14, 32 if jammer else 14)
        ax.grid(False); ax.set_ylabel("y [m]")
        ax.text(5, -11.5, "eastbound →", fontsize=7.5, color="#52514e", va="center")
        ax.text(1995, 11.5, "← westbound", fontsize=7.5, color="#52514e", va="center", ha="right")
        if jammer:
            ax.scatter([jammer[0]], [jammer[1]], marker="X", s=140, color="#1f1e1c", zorder=5,
                       label="jammer (jamming condition only)")
        ax.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.005, 1.0))
    axes[-1, 0].set_xlabel("x [m]  (vertical axis stretched)")
    fig.suptitle(SCENARIO_LABEL[scenario], x=0.01, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()

def urban_map(scenario="urban_intersection"):
    lvls = levels(scenario)
    fig, axes = plt.subplots(1, len(lvls), figsize=(7.2 * len(lvls), 8.2), squeeze=False)
    for ax, level in zip(axes[0], lvls):
        draw_scenario(ax, scenario, level)
        ax.set_xlim(0, 800); ax.set_ylim(0, 800); ax.set_aspect("equal"); ax.grid(False)
        ax.set_xlabel("x [m]")
        ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)
    axes[0][0].set_ylabel("y [m]")
    fig.suptitle(SCENARIO_LABEL[scenario] + " (buildings in dark grey)", x=0.01, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()

jam = cfg["scenarios"]["highway_platoon"]["conditions"]["jamming"]["arguments"]["Jammer"].split(",")
# highway_map("highway_platoon", jammer=(float(jam[0]), float(jam[1])))
highway_map("highway_platoon")
urban_map()
highway_map("highway_cpm")

### 1.5 Mechanisms as they appear in the data

In `highway_platoon` and `urban_intersection` each flow keeps its own scheduling from the table
above (periodic flows on SB-SPS, emergency and crash mitigation on DS), so the rows of those
figures differ by re-evaluation and pre-emption only. `highway_cpm` also switches the CPMs between
SB-SPS and DS. On DS the CPM never announces a future resource, so pre-emption has nothing to act
on there: DS + re-evaluation + pre-emption equals DS + re-evaluation.

In [ ]:
def mechanisms(scenario):
    """(label, variant, policy) of every mechanism of a scenario, in display order."""
    if scenario == "highway_cpm":
        return [("SB-SPS", "sps_max", "off"), ("SB-SPS + re-evaluation", "sps_max", "reeval"),
                ("SB-SPS + re-eval + pre-emption", "sps_max", "reeval_preempt"),
                ("DS", "dynamic", "off"), ("DS + re-evaluation", "dynamic", "reeval"),
                ("DS + re-eval + pre-emption", "dynamic", "reeval_preempt")]
    return [("sensing only", "default", "off"), ("+ re-evaluation", "default", "reeval"),
            ("+ re-eval + pre-emption", "default", "reeval_preempt")]

def pick(frame, scenario, variant, policy, condition="baseline"):
    return frame[(frame.scenario == scenario) & (frame.variant == variant) & (frame.policy == policy)
                 & (frame.condition == condition)]

pd.DataFrame([{"scenario": sc, "mechanism": m, "variant": v, "policy": p,
               "runs": pick(flows, sc, v, p).id.nunique()} for sc in SCENARIO_ORDER for m, v, p in mechanisms(sc)])

## 2. Can the flows coexist? Metrics against the Tx-Rx distance

One figure per scenario and metric. Rows are the mechanisms, columns the background levels, one
line per flow (the mean over the 3 seeds), the band its 95 % confidence interval. Every metric
here is a proportion, so the interval is a t interval over the seeds on the logit scale, mapped
back: with only 3 seeds (t = 4.30) a plain mean ± t·s/√3 would cross 0 or 1 wherever the seeds
disagree near a bound, and show impossible values such as a negative PDR. Bins of 50 m, a point is
left out when a run has fewer than 50 attempts in the bin.

* **QoS-PDR**: decoded within the PDB of the flow. **PDR**: decoded at all.
* **PER** = 1 − QoS-PDR on a log scale, with one dashed line per PER target, labelled with the flows
  it applies to: a flow meets its requirement at a distance only where its line is under its
  target. On this log scale the line is the geometric mean over the seeds and the band its
  interval on log10(PER); a seed with no failure in a bin counts as half an attempt failed.
* **collision rate**: attempts lost to a collision; **hidden-node collision rate**: the part of
  those where the interferer was hidden from the sender (no packet of it decoded in the previous
  1.1 s).

In [ ]:
DIST_METRICS = [
    ("qos_pdr", "QoS-PDR", "linear"),
    ("pdr", "PDR", "linear"),
    ("per", "PER = 1 − QoS-PDR", "log"),
    ("collision", "collision rate", "linear"),
    ("hidden", "hidden-node collision rate", "linear"),
]
bd = by_distance[by_distance.condition == "baseline"].copy()
bd["per"] = 1 - bd.qos_pdr
bd["x"] = bd["bin"] + 25

def distance_figure(scenario, metric, label, scale):
    mechs, lvls = mechanisms(scenario), levels(scenario)
    fig, axes = plt.subplots(len(mechs), len(lvls), figsize=(5.4 * len(lvls), 2.5 * len(mechs) + 0.8),
                             sharex=True, sharey=True, squeeze=False)
    names = ordered(bd[bd.scenario == scenario].name.unique())
    for r, (mech, variant, policy) in enumerate(mechs):
        part = pick(bd, scenario, variant, policy)
        part = part[part.attempts >= MIN_ATTEMPTS]
        for c, level in enumerate(lvls):
            ax = axes[r][c]
            lp = part[part.density == level]
            for name in names:
                col, mk, ls = FLOW_STYLE[name]
                rows_f = lp[lp.name == name].copy()
                if scale == "log":
                    # geometric mean and interval on log10, a zero PER counts as half an attempt
                    rows_f["lv"] = np.log10(np.maximum(rows_f[metric], 0.5 / rows_f.attempts))
                    stats = ci95_by(rows_f, "x", "lv")
                    if stats.empty:
                        continue
                    h = stats["h"].fillna(0)
                    m, lo, hi = 10 ** stats["m"], 10 ** (stats["m"] - h), 10 ** (stats["m"] + h)
                else:
                    # every linear metric here is a proportion: the band is computed on the logit
                    # scale so that it stays inside [0, 1]; the line is the plain mean
                    floor = 0.5 / rows_f.attempts
                    rows_f["lv"] = logit(rows_f[metric].clip(lower=floor, upper=1 - floor))
                    stats = ci95_by(rows_f, "x", "lv")
                    if stats.empty:
                        continue
                    h = stats["h"].fillna(0)
                    m = rows_f.groupby("x")[metric].mean().reindex(stats.index)
                    lo, hi = expit(stats["m"] - h), expit(stats["m"] + h)
                ax.plot(stats.index, m, color=col, ls=ls, marker=mk, markersize=4, markevery=2, lw=1.6)
                ax.fill_between(stats.index, lo, hi, color=col, alpha=0.16, lw=0)
            if metric == "per":
                targets = {}
                for name in names:
                    targets.setdefault(classes[name]["per"], []).append(FLOW_LABEL[name].split(" (")[0].split(",")[0])
                for target, who in targets.items():
                    ax.axhline(target, color="#52514e", ls=(0, (4, 3)), lw=0.9)
                    ax.text(985, target * 1.25, "target %.0e: %s" % (target, ", ".join(who)), ha="right", va="bottom",
                            fontsize=6.5, color="#52514e")
            if scale == "log":
                ax.set_yscale("log"); ax.set_ylim(3e-6, 1.5)
            if r == 0:
                ax.set_title(level_label(scenario, level), fontsize=9.5)
            if c == 0:
                ax.set_ylabel(mech + "\n" + label, fontsize=8.5)
            if r == len(mechs) - 1:
                ax.set_xlabel("distance from the Tx [m]")
    handles = [Line2D([], [], color=color(n), ls=FLOW_STYLE[n][2], marker=FLOW_STYLE[n][1], label=FLOW_LABEL[n]) for n in names]
    if metric == "per":
        handles.append(Line2D([], [], color="#52514e", ls=(0, (4, 3)), lw=1, label="PER target"))
    fig.legend(handles=handles, loc="upper center", ncol=min(4, len(handles)), fontsize=8, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle("%s: %s against distance" % (SCENARIO_LABEL[scenario], label), y=1.04, x=0.02, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()

### 2.1 Highway platooning

In [ ]:
for metric, label, scale in DIST_METRICS:
    distance_figure("highway_platoon", metric, label, scale)

### 2.2 Urban intersections

In [ ]:
for metric, label, scale in DIST_METRICS:
    distance_figure("urban_intersection", metric, label, scale)

### 2.3 Highway CPM contention

In [ ]:
for metric, label, scale in DIST_METRICS:
    distance_figure("highway_cpm", metric, label, scale)

### 2.4 Verdict per flow

Measured PER within `RANGE_M` (attempts pooled over the bins, mean over the seeds) divided by the
target of the flow. Under 1 the flow meets its requirement on that range, the cell is marked ✓.

In [ ]:
def near(frame, max_m=RANGE_M):
    x = frame[frame["bin"] < max_m].copy()
    x["ok"] = x.qos_pdr * x.attempts
    x["dec"] = x.pdr * x.attempts
    x["col"] = x.collision * x.attempts
    x["hid"] = x.hidden * x.attempts
    keys = ["id", "scenario", "density", "condition", "variant", "policy", "seed", "name"]
    g = x.groupby(keys)[["attempts", "ok", "dec", "col", "hid"]].sum().reset_index()
    g["qos_pdr"] = g.ok / g.attempts; g["pdr"] = g.dec / g.attempts
    g["collision"] = g.col / g.attempts; g["hidden"] = g.hid / g.attempts
    return g

NEAR = near(by_distance)
rows = []
for scenario in SCENARIO_ORDER:
    for mech, variant, policy in mechanisms(scenario):
        part = pick(NEAR, scenario, variant, policy)
        for (level, name), g in part.groupby(["density", "name"]):
            per = (1 - g.qos_pdr).mean()
            rows.append({"scenario": scenario, "mechanism": mech, "background": level, "flow": name,
                         "PER / target": per / classes[name]["per"]})
verdict = pd.DataFrame(rows)
table = verdict.pivot_table(index=["scenario", "mechanism"], columns=["flow", "background"], values="PER / target", sort=False)
table = table.reindex(columns=[(f, l) for f in FLOW_ORDER for l in LEVEL_ORDER if (f, l) in table.columns])
table.style.format(lambda v: "" if pd.isna(v) else ("✓ %.2g" % v if v <= 1 else "%.3g" % v)).background_gradient(
    cmap="Greys", axis=None, vmin=0, vmax=np.nanpercentile(table.values, 90))

## 3. Why packets are lost

Each failed attempt has exactly one cause (notebook 10, section 7):

| Cause | Meaning |
|---|---|
| late | decoded, but after the PDB |
| oversize drop, intra-UE conflict, never transmitted | the packet never left the MAC (too large, another flow of the UE had the slot, no resource in time) |
| half duplex | the receiver was transmitting in that slot |
| below sensitivity, propagation | the signal alone was too weak (path loss, shadowing, buildings) |
| hidden-node collision | lost to interference from a UE the sender could not sense |
| sensing-visible collision | lost to interference from a UE the sender could sense, so sensing could in principle have avoided it |
| injected burst | an injected Gilbert-Elliott loss burst (impairment condition) |

The boxes give the share of the failed attempts within `RANGE_M` that each cause explains, one
sample per run (3 seeds × every mechanism of the scenario, baseline condition). The whiskers span
1.5 × the inter-quartile range. Superseded CPMs are not failures and do not appear here (section 1.1).

In [ ]:
CAUSES = list(plots.CAUSE_ORDER)
fails = causes[(causes.condition == "baseline") & (causes["bin"] < RANGE_M) & (causes.cause != "decoded in time")]

def cause_boxes(scenario):
    variants = {v for _, v, _ in mechanisms(scenario)}
    part = fails[(fails.scenario == scenario) & fails.variant.isin(variants)]
    share = part.groupby(["density", "id", "name", "cause"])["count"].sum().unstack(fill_value=0)
    share = 100 * share.div(share.sum(axis=1), axis=0)
    used = [c for c in CAUSES if c in share.columns and share[c].max() > 0.5]
    share = share[used]
    lvls, names = levels(scenario), ordered(part.name.unique())
    fig, axes = plt.subplots(1, len(lvls), figsize=(7.5 * len(lvls), 4.2), sharey=True, squeeze=False)
    width = 0.8 / len(names)
    for ax, level in zip(axes[0], lvls):
        data = share.xs(level, level="density")
        for k, name in enumerate(names):
            d = data.xs(name, level="name") if name in data.index.get_level_values("name") else None
            if d is None:
                continue
            pos = np.arange(len(used)) - 0.4 + width * (k + 0.5)
            col = color(name)
            bp = ax.boxplot([d[c].values for c in used], positions=pos, widths=width * 0.85, patch_artist=True,
                            showfliers=True, flierprops=dict(marker=FLOW_STYLE[name][1], markersize=3, markerfacecolor=col, markeredgecolor="none"),
                            medianprops=dict(color="#1f1e1c", lw=1.2), whiskerprops=dict(color=col), capprops=dict(color=col))
            for box in bp["boxes"]:
                box.set(facecolor=col, alpha=0.75, edgecolor=col)
        ax.set_xticks(range(len(used)))
        ax.set_xticklabels([c.replace(" (decoded after the PDB)", "").replace(" collision", "\ncollision").replace(" by a newer", "\nby a newer")
                            for c in used], fontsize=8)
        for x in np.arange(len(used) - 1) + 0.5:
            ax.axvline(x, color="#e6e5e0", lw=0.8)
        ax.set_title(level_label(scenario, level), fontsize=9.5)
    axes[0][0].set_ylabel("share of failed attempts within %d m [%%]" % RANGE_M)
    fig.legend(handles=[Patch(facecolor=color(n), alpha=0.75, label=FLOW_LABEL[n]) for n in names],
               loc="upper center", ncol=min(4, len(names)), fontsize=8, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle("%s: loss causes" % SCENARIO_LABEL[scenario], y=1.09, x=0.02, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()
    loss = pick(NEAR, scenario, *mechanisms(scenario)[0][1:]).groupby(["density", "name"]).qos_pdr.mean()
    return (100 * (1 - loss)).unstack("density").reindex(names).rename(index=FLOW_LABEL).round(2).rename_axis(
        columns="failed attempts within %d m [%%], %s" % (RANGE_M, mechanisms(scenario)[0][0]))

for scenario in SCENARIO_ORDER:
    display(cause_boxes(scenario))

## 4. PDR against latency, and the requirement of each flow

One point per flow and background level: PDR within `RANGE_M` (decoded at all) against the
latency of the decoded packets. The marker sits at the median latency, with its 95 % interval over
the seeds; the thin line of the same colour runs on to the 99th percentile. The vertical bar is the
95 % interval of the PDR, computed on log10(1 − PDR) like the PER above. The colour is the flow,
the marker the background level. The light disc of the flow colour is centred on its requirement
(PDB, 1 − PER target): **a flow meets it when its marker and the end of its line are to the left
of the disc centre and its marker is above it.** Two flows with the same requirement (CAM and VAM)
share the disc, one half each. The PDR axis is stretched near 1 so that 0.99,
0.999, ... are evenly spaced; the latency axis is logarithmic.

The MAC never schedules a transmission after the PDB, so the 99th percentile latency always ends
at or just under the PDB: no flow loses packets for being late, they are lost on the air.

In [ ]:
def nines(p): return -np.log10(np.clip(1 - np.asarray(p, dtype=float), 1e-7, 1))
def un_nines(n): return 1 - 10 ** (-np.asarray(n, dtype=float))
NINE_TICKS = [0.5, 0.9, 0.99, 0.999, 0.9999, 0.99999]
latency = flows[flows.condition == "baseline"][["id", "name", "latency_p99_ms", "latency_p50_ms"]]
pts = NEAR[NEAR.condition == "baseline"].merge(latency, on=["id", "name"])

def requirement_figure(scenario):
    mechs, lvls = mechanisms(scenario), levels(scenario)
    names = ordered(pts[pts.scenario == scenario].name.unique())
    fig, axes = plt.subplots(1, len(mechs), figsize=(4.3 * len(mechs), 4.4), sharey=True, sharex=True, squeeze=False)
    for ax, (mech, variant, policy) in zip(axes[0], mechs):
        part = pick(pts, scenario, variant, policy)
        shared = {}
        for name in names:
            shared.setdefault((classes[name]["pdb_ms"], classes[name]["per"]), []).append(name)
        for (pdb, per), who in shared.items():
            # one disc per requirement; two flows with the same requirement get one half each
            fills = ["full"] if len(who) != 2 else ["left", "right"]
            for name, fill in zip(who, fills * len(who)):
                ax.plot([pdb], [1 - per], ls="", marker="o", markersize=33, fillstyle=fill, color=color(name),
                        markerfacecoloralt=(1, 1, 1, 0), markeredgewidth=0, alpha=0.3, zorder=1)
        for name in names:
            col = color(name)
            for level in lvls:
                g = part[(part.name == name) & (part.density == level)]
                if g.empty:
                    continue
                x, xh = ci95(g.latency_p50_ms); x99, _ = ci95(g.latency_p99_ms)
                lv, lh = ci95(np.log10(np.maximum(1 - g.pdr, 0.5 / g.attempts)))
                lh = 0 if np.isnan(lh) else lh
                y, ylo, yhi = 1 - 10 ** lv, 1 - 10 ** (lv + lh), 1 - 10 ** (lv - lh)
                ax.plot([x, x99], [y, y], color=col, lw=1.2, alpha=0.55, zorder=2, solid_capstyle="butt")
                ax.errorbar(x, y, xerr=[[min(xh, x * 0.9)], [xh]] if not np.isnan(xh) else None,
                            yerr=[[y - ylo], [yhi - y]], fmt=LEVEL_MARKER[level], color=col,
                            markersize=7, markeredgecolor="white", markeredgewidth=0.8, elinewidth=1, capsize=2, zorder=3)
        ax.set_xscale("log"); ax.set_yscale("function", functions=(nines, un_nines))
        ax.set_yticks(NINE_TICKS); ax.set_yticklabels([str(t) for t in NINE_TICKS])
        ax.set_ylim(0.3, 0.999995); ax.set_xlim(0.3, 200)
        ax.set_xticks([0.3, 1, 3, 10, 20, 50, 100]); ax.set_xticklabels(["0.3", "1", "3", "10", "20", "50", "100"])
        ax.set_title(mech, fontsize=9.5); ax.set_xlabel("latency [ms]: marker median, line to p99")
    axes[0][0].set_ylabel("PDR within %d m" % RANGE_M)
    handles = [Patch(facecolor=color(n), label=FLOW_LABEL[n]) for n in names]
    handles += [Line2D([], [], ls="", marker=LEVEL_MARKER[l], color="#52514e", markersize=7, label=level_label(scenario, l)) for l in lvls]
    handles += [Line2D([], [], ls="", marker="o", color="#8a8985", alpha=0.3, markersize=16, label="requirement (PDB, 1 − PER)")]
    fig.legend(handles=handles, loc="upper center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle("%s: PDR against latency" % SCENARIO_LABEL[scenario], y=1.12, x=0.02, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()

for scenario in SCENARIO_ORDER:
    requirement_figure(scenario)

## 5. What re-evaluation and pre-emption bring

Paired differences on the same seed (same trace), in percentage points of QoS-PDR within
`RANGE_M` unless stated, mean and 95 % confidence interval over the 3 seeds:

* **re-evaluation** = re-evaluation − sensing only
* **pre-emption** = (re-evaluation + pre-emption) − re-evaluation, the added value of pre-emption
* **both** = (re-evaluation + pre-emption) − sensing only

A bar whose interval does not cross 0 is a change the 3 seeds agree on.

### 5.1 Overall

"Overall" weights every flow of the scenario equally (mean of the per-flow values), so a rare
flow counts as much as the CAMs. Next to QoS-PDR: the collision rate, the mean age of information
and the 99th percentile disconnection time (all receivers within 1000 m). AoI and DT leave crash
mitigation out: its packets come in rare bursts (one event per vehicle every 20 s on average), so
its gaps are the time between events, not disconnections. The same holds for its AoI and DT
columns in the per-flow tables.

In [ ]:
CONTRIB = [("re-evaluation", "reeval", "off"), ("pre-emption", "reeval_preempt", "reeval"),
           ("both", "reeval_preempt", "off")]
CONTRIB_STYLE = {"re-evaluation": dict(facecolor="#52514e", hatch=""),
                 "pre-emption": dict(facecolor="white", edgecolor="#52514e", hatch="////"),
                 "both": dict(facecolor="#b8b7b0", hatch="")}
per_flow = NEAR.merge(flows[["id", "name", "aoi_mean_ms", "dt_p99_ms", "collision_rate"]], on=["id", "name"])
per_flow["qos_pdr_pp"] = 100 * per_flow.qos_pdr
per_flow["collision_pp"] = 100 * per_flow.collision
KEYS = ["scenario", "density", "condition", "variant", "seed"]

def deltas(frame, value, extra=()):
    out = []
    keys = KEYS + list(extra)
    for label, a, b in CONTRIB:
        A = frame[frame.policy == a].set_index(keys)[value]
        B = frame[frame.policy == b].set_index(keys)[value]
        d = (A - B).dropna().rename("delta").reset_index(); d["contribution"] = label
        out.append(d)
    return pd.concat(out, ignore_index=True)

SHORT = {"highway_platoon": "Platooning", "urban_intersection": "Urban", "highway_cpm": "CPM highway"}
def group_label(scenario, density, variant):
    tail = {"sps_max": "\nCPM on SB-SPS", "dynamic": "\nCPM on DS"}.get(variant, "")
    return "%s\n%s background%s" % (SHORT[scenario], density, tail)

# AoI and DT of an event flow measure the time between events, not a disconnection
EVENT_FLOWS = [n for n, c in classes.items() if c["model"] == "event"]

base = per_flow[(per_flow.condition == "baseline") & (per_flow.variant != "sps_median")]
OVERALL_METRICS = [("qos_pdr_pp", "Δ QoS-PDR within %d m [pp]" % RANGE_M), ("collision_pp", "Δ collision rate [pp]"),
                   ("aoi_mean_ms", "Δ mean AoI [ms]"), ("dt_p99_ms", "Δ DT p99 [ms]")]
overall = base.groupby(KEYS + ["policy"])[["qos_pdr_pp", "collision_pp"]].mean()
overall = overall.join(base[~base.name.isin(EVENT_FLOWS)].groupby(KEYS + ["policy"])[["aoi_mean_ms", "dt_p99_ms"]].mean()).reset_index()
groups = [(sc, d, v) for sc in SCENARIO_ORDER for d in levels(sc) for v in sorted({v for _, v, _ in mechanisms(sc)}, reverse=True)]

fig, axes = plt.subplots(len(OVERALL_METRICS), 1, figsize=(14, 3.0 * len(OVERALL_METRICS)), sharex=True)
for ax, (metric, label) in zip(axes, OVERALL_METRICS):
    d = deltas(overall, metric)
    for k, (clabel, _, _) in enumerate(CONTRIB):
        ms, hs = [], []
        for sc, dens, var in groups:
            m, h = ci95(d[(d.contribution == clabel) & (d.scenario == sc) & (d.density == dens) & (d.variant == var)].delta)
            ms.append(m); hs.append(h)
        ax.bar(np.arange(len(groups)) + 0.27 * (k - 1), ms, 0.25, yerr=hs, capsize=2, label=clabel,
               edgecolor=CONTRIB_STYLE[clabel].get("edgecolor", "none"), facecolor=CONTRIB_STYLE[clabel]["facecolor"],
               hatch=CONTRIB_STYLE[clabel]["hatch"], error_kw=dict(ecolor="#1f1e1c", lw=0.8))
    ax.axhline(0, color="#1f1e1c", lw=0.8); ax.set_ylabel(label, fontsize=8.5)
axes[-1].set_xticks(range(len(groups))); axes[-1].set_xticklabels([group_label(*g) for g in groups], fontsize=7.5)
axes[0].legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.25))
axes[0].set_title("Overall contribution (mean over the flows of the scenario)", y=1.2)
plt.tight_layout(); plt.show()

### 5.2 Per flow

Same three contributions per flow, on QoS-PDR within `RANGE_M`. The bar keeps the flow colour,
the hatch tells the contribution: solid = re-evaluation, hatched = pre-emption, pale = both.
The table under each figure gives the four metrics; ★ marks an interval that excludes 0.

In [ ]:
FLOW_HATCH = {"re-evaluation": dict(alpha=1.0, hatch=""), "pre-emption": dict(alpha=1.0, hatch="////", fill=False),
              "both": dict(alpha=0.35, hatch="")}

def flow_contribution(scenario):
    lvls = levels(scenario)
    variants = sorted({v for _, v, _ in mechanisms(scenario)}, reverse=True)
    part = base[base.scenario == scenario]
    names = ordered(part.name.unique())
    d = deltas(part, "qos_pdr_pp", extra=["name"])
    cols = [(l, v) for v in variants for l in lvls]
    fig, axes = plt.subplots(1, len(cols), figsize=(max(4.2, 1.15 * len(names)) * len(cols), 3.6), sharey=True, squeeze=False)
    for ax, (level, variant) in zip(axes[0], cols):
        for i, name in enumerate(names):
            for k, (clabel, _, _) in enumerate(CONTRIB):
                m, h = ci95(d[(d.contribution == clabel) & (d.density == level) & (d.variant == variant) & (d.name == name)].delta)
                st = FLOW_HATCH[clabel]
                ax.bar(i + 0.27 * (k - 1), m, 0.25, yerr=h, capsize=2, color=color(name), edgecolor=color(name),
                       alpha=st["alpha"], hatch=st["hatch"], fill=st.get("fill", True), lw=1,
                       error_kw=dict(ecolor="#1f1e1c", lw=0.8))
        ax.axhline(0, color="#1f1e1c", lw=0.8)
        ax.set_xticks(range(len(names))); ax.set_xticklabels([FLOW_LABEL[n].split(" (")[0] for n in names], rotation=25, ha="right", fontsize=8)
        tail = {"sps_max": ", CPM on SB-SPS", "dynamic": ", CPM on DS"}.get(variant, "")
        ax.set_title(level_label(scenario, level) + tail, fontsize=9)
    axes[0][0].set_ylabel("Δ QoS-PDR within %d m [pp]" % RANGE_M)
    fig.legend(handles=[Patch(facecolor="#52514e", label="re-evaluation"),
                        Patch(facecolor="white", edgecolor="#52514e", hatch="////", label="pre-emption (on top of re-evaluation)"),
                        Patch(facecolor="#52514e", alpha=0.35, label="both")],
               loc="upper center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle("%s: contribution per flow" % SCENARIO_LABEL[scenario], y=1.1, x=0.02, ha="left", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

    rows = []
    for metric, label in OVERALL_METRICS:
        dm = deltas(part, metric, extra=["name"])
        for (level, variant, name, clabel), g in dm.groupby(["density", "variant", "name", "contribution"]):
            m, h = ci95(g.delta)
            star = "★" if not np.isnan(h) and abs(m) > h else ""
            rows.append({"background": level, "variant": variant, "flow": FLOW_LABEL[name], "metric": label,
                         "contribution": clabel, "value": "%+.2f ± %.2f %s" % (m, h, star)})
    t = pd.DataFrame(rows).pivot_table(index=["variant", "background", "flow"], columns=["metric", "contribution"],
                                       values="value", aggfunc="first", sort=False)
    t = t.reindex(columns=[(label, c) for _, label in OVERALL_METRICS for c, _, _ in CONTRIB])
    return t

for scenario in SCENARIO_ORDER:
    display(flow_contribution(scenario))

### 5.3 How often the mechanisms act

Re-evaluation checks and the resources they actually moved, pre-emption checks and moves, per
1000 transmissions of the flow, counted on the flow whose resource was checked. With re-evaluation
off MoReV2X still runs the check and logs what it would have moved, without moving it: those
counterfactual moves are shown for the "sensing only" rows.

In [ ]:
acts = flows[(flows.condition == "baseline") & (flows.variant != "sps_median")].copy()
for col in ["reeval_checks", "reeval_changes", "preempt_checks", "preempt_changes"]:
    acts[col + "_pk"] = 1000 * acts[col] / acts.transmissions.clip(lower=1)
rows = []
for scenario in SCENARIO_ORDER:
    for mech, variant, policy in mechanisms(scenario):
        g = pick(acts, scenario, variant, policy).groupby(["density", "name"])[
            ["reeval_checks_pk", "reeval_changes_pk", "preempt_checks_pk", "preempt_changes_pk"]].mean()
        g["scenario"], g["mechanism"] = scenario, mech
        rows.append(g.reset_index())
acts_t = pd.concat(rows)
acts_t["name"] = acts_t.name.map(FLOW_LABEL)
acts_t.set_index(["scenario", "density", "mechanism", "name"]).round(1).rename(columns=lambda c: c.replace("_pk", " per 1000 tx"))

## 6. More from the data

### 6.1 Reliable range

The largest distance up to which the mean QoS-PDR of every 50 m bin stays at or above a level:
the flow's own target (1 − PER), 0.99 and 0.9. 0 means the first bin already misses it.

In [ ]:
def reliable_range(stats, level):
    ok = (stats >= level).values
    n = 0
    while n < len(ok) and ok[n]:
        n += 1
    return 50 * n

rows = []
for scenario in SCENARIO_ORDER:
    for mech, variant, policy in mechanisms(scenario):
        part = pick(bd, scenario, variant, policy)
        for (level, name), g in part.groupby(["density", "name"]):
            stats = g.groupby("bin").qos_pdr.mean().sort_index()
            rows.append({"scenario": scenario, "mechanism": mech, "background": level, "flow": name,
                         "target": reliable_range(stats, 1 - classes[name]["per"]),
                         "0.99": reliable_range(stats, 0.99), "0.9": reliable_range(stats, 0.9)})
rr = pd.DataFrame(rows)
rr_t = rr.pivot_table(index=["scenario", "mechanism"], columns=["flow", "background"], values=["0.99", "0.9"], sort=False)
rr_t.style.format("{:.0f}").background_gradient(cmap="Blues", axis=None)

### 6.2 Channel load

Mean channel busy ratio (CBR) per scenario, background level and mechanism, with the 95 %
interval over the seeds. Re-evaluation and pre-emption move resources, they do not add
transmissions, but a moved resource is a second reservation on air for a while.

In [ ]:
runs = flows.drop_duplicates("id")
runs = runs[(runs.condition == "baseline") & (runs.variant != "sps_median")]
fig, axes = plt.subplots(1, len(SCENARIO_ORDER), figsize=(15, 4.4), sharey=True)
for ax, scenario in zip(axes, SCENARIO_ORDER):
    mechs, lvls = mechanisms(scenario), levels(scenario)
    width = 0.8 / len(mechs)
    greys = plt.cm.Greys(np.linspace(0.35, 0.9, len(mechs)))
    for k, (mech, variant, policy) in enumerate(mechs):
        stats = [ci95(pick(runs, scenario, variant, policy).query("density == @l").cbr_mean) for l in lvls]
        ax.bar(np.arange(len(lvls)) + width * (k - (len(mechs) - 1) / 2), [m for m, _ in stats], width * 0.92,
               yerr=[h for _, h in stats], color=greys[k], label=mech, capsize=2, hatch="//" if variant == "dynamic" else "")
    ax.set_xticks(range(len(lvls))); ax.set_xticklabels([level_label(scenario, l).replace(" background", "\nbackground") for l in lvls], fontsize=8)
    ax.set_title(SCENARIO_LABEL[scenario]); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2)
axes[0].set_ylabel("mean CBR"); plt.tight_layout(); plt.show()

### 6.3 Platooning under impairments

`highway_platoon` also ran with a wideband jammer, Gilbert-Elliott loss bursts at the platoon
trucks, and 3 to 7 ms of extra higher-layer latency on the platooning messages. The figure gives
the platooning flow only, per condition and mechanism: QoS-PDR within 150 m (inside a platoon of
six trucks), and over all receivers the 99th percentile disconnection time and the mean age of
information.

In [ ]:
plat = near(by_distance[(by_distance.scenario == "highway_platoon") & (by_distance.name == "platooning")], 150)
plat = plat.merge(flows[["id", "name", "dt_p99_ms", "aoi_mean_ms", "pir_p99_ms"]], on=["id", "name"])
plat["qos_pdr_pct"] = 100 * plat.qos_pdr
CONDITIONS = ["baseline", "jamming", "loss_bursts", "stack_latency"]
mechs = mechanisms("highway_platoon")
fig, axes = plt.subplots(len(levels("highway_platoon")), 3, figsize=(15, 3.2 * len(levels("highway_platoon"))), squeeze=False)
greys = plt.cm.Greys(np.linspace(0.35, 0.9, len(mechs)))
for r, level in enumerate(levels("highway_platoon")):
    for c, (metric, label) in enumerate([("qos_pdr_pct", "QoS-PDR within 150 m [%]"), ("dt_p99_ms", "DT p99 [ms]"),
                                         ("aoi_mean_ms", "mean AoI [ms]")]):
        ax = axes[r][c]
        for k, (mech, variant, policy) in enumerate(mechs):
            sel = [plat[(plat.density == level) & (plat.condition == cond) & (plat.policy == policy)] for cond in CONDITIONS]
            if metric == "qos_pdr_pct":
                stats = [proportion_ci(g.qos_pdr, g.attempts) for g in sel]
                means = [100 * m for m, _, _ in stats]
                err = [[100 * (m - lo) for m, lo, _ in stats], [100 * (hi - m) for m, _, hi in stats]]
            else:
                stats = [ci95(g[metric]) for g in sel]
                means, err = [m for m, _ in stats], [h for _, h in stats]
            ax.bar(np.arange(len(CONDITIONS)) + 0.27 * (k - 1), means, 0.25, yerr=err,
                   color=greys[k], label=mech, capsize=2, error_kw=dict(lw=0.8))
        ax.set_xticks(range(len(CONDITIONS))); ax.set_xticklabels(CONDITIONS, fontsize=8)
        ax.set_ylabel(label, fontsize=8.5); ax.set_title(level_label("highway_platoon", level), fontsize=9)
        if metric == "qos_pdr_pct":
            ax.set_ylim(90, 100.2)
fig.legend(*axes[0][0].get_legend_handles_labels(), loc="upper center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Platooning (PQI 21) under impairments", x=0.02, ha="left", y=1.03, fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

### 6.4 CPMs: SB-SPS against DS

`highway_cpm` with every vehicle sending CAMs and CPMs. Paired difference DS − SB-SPS (same seed and
policy) for both flows, in QoS-PDR within `RANGE_M`, collision rate, mean AoI and channel
occupancy (share of the subchannel-slots the flow used).

In [ ]:
cpm = per_flow[(per_flow.scenario == "highway_cpm") & (per_flow.condition == "baseline")].merge(
    flows[["id", "name", "occupancy"]], on=["id", "name"])
keys = ["density", "policy", "seed", "name"]
rows = []
for metric, label in [("qos_pdr_pp", "Δ QoS-PDR [pp]"), ("collision_pp", "Δ collision [pp]"),
                      ("aoi_mean_ms", "Δ mean AoI [ms]"), ("occupancy", "Δ occupancy")]:
    A = cpm[cpm.variant == "dynamic"].set_index(keys)[metric]
    B = cpm[cpm.variant == "sps_max"].set_index(keys)[metric]
    for (level, policy, name), g in (A - B).dropna().groupby(["density", "policy", "name"]):
        m, h = ci95(g)
        rows.append({"background": level, "policy": plots.POLICY_LABELS[policy], "flow": FLOW_LABEL[name], "metric": label,
                     "DS − SB-SPS": "%+.2f ± %.2f%s" % (m, h, " ★" if abs(m) > h else "")})
pd.DataFrame(rows).pivot_table(index=["background", "policy", "flow"], columns="metric", values="DS − SB-SPS", aggfunc="first")

### 6.5 Disconnection tails

The disconnection time (DT) is how long a receiver stays without news from a sender after a loss
(Rolich et al.). Its complementary CDF per flow, all receivers and seeds of the densest background,
for the three mechanisms (line style = mechanism here, colour = flow). The histograms stop at
2 s, so a curve ends there with everything longer. Crash mitigation is left out (event traffic).
CPMs are missing until the sweep is rerun: their disconnection times counted superseded CPMs as
losses, and the raw logs needed to recompute them are gone (`tools/qos/exclude_superseded.py`).

In [ ]:
MECH_LS = ["-", "--", ":"]
for scenario in SCENARIO_ORDER:
    level = levels(scenario)[-1]
    h = histograms[(histograms.metric == "dt") & (histograms.scenario == scenario) & (histograms.density == level)
                   & (histograms.condition == "baseline")]
    fig, ax = plt.subplots(figsize=(11, 3.4))
    for k, (mech, variant, policy) in enumerate(mechanisms(scenario)[:3]):
        part = h[(h.variant == variant) & (h.policy == policy)]
        for name in [n for n in ordered(part.name.unique()) if n not in EVENT_FLOWS]:
            g = part[part.name == name].groupby("edge_s")["count"].sum().sort_index()
            ccdf = 1 - g.cumsum() / g.sum()
            ax.step(1000 * g.index, ccdf.clip(lower=1e-5), where="post", color=color(name), ls=MECH_LS[k], lw=1.4)
    ax.set_yscale("log"); ax.set_xlim(0, 2000); ax.set_ylim(1e-4, 1)
    ax.set_xlabel("disconnection time [ms]"); ax.set_ylabel("P(DT > x)")
    ax.set_title("%s, %s" % (SCENARIO_LABEL[scenario], level_label(scenario, level)), fontsize=10)
    names = [n for n in ordered(h.name.unique()) if n not in EVENT_FLOWS]
    ax.legend(handles=[Line2D([], [], color=color(n), label=FLOW_LABEL[n]) for n in names] +
              [Line2D([], [], color="#52514e", ls=MECH_LS[k], label=m) for k, (m, _, _) in enumerate(mechanisms(scenario)[:3])],
              fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.show()

## 7. What the data says

Written from `qos_full` (144 runs, 3 seeds). Numbers are within 300 m unless stated.

**Coexistence: no.** None of the 90 cases (flow × scenario × background × mechanism) meets its
PER target within 300 m; the closest is advanced driving at low background, 3.5 times its 10⁻²
target. Even near the sender the strict flows stay far off: platooning and crash mitigation reach
a QoS-PDR of about 0.998 within 150 m at low background, a PER about 14 and 140 times their 10⁻⁴
and 10⁻⁵ targets. CPMs reach a QoS-PDR of 0.76 to 0.92 within 300 m on the highways (SB-SPS),
against a target of 0.99. With high background every flow loses several points, and on the
highways 86 to 97 % of every flow's failures are sensing-visible collisions: the flows collide with
transmissions their sender could hear. On a 20 MHz channel of two subchannels every CAM above
about 300 B, every CPM, platooning and emergency message takes the whole slot, so the flows compete
for slots rather than subchannels, and the CBR reaches 0.77 to 0.81 on the dense highway.

**Re-evaluation** helps on the highway: +0.7 to +1.5 points of QoS-PDR within 300 m for
platooning, advanced driving and CAM at high background, +3.3 to +4.0 points for the CPMs on
SB-SPS, and 1 to 2 points of collisions less. It costs crash mitigation (DS, the highest priority)
1.2 points on the dense platooning highway, and gives CPMs on DS nothing (−0.7 to −1.0 points).

**Pre-emption** (on top of re-evaluation) helps the highest priority flow that holds SB-SPS
reservations: platooning gains 4.8 points on the dense highway. The others pay for it: CPMs, CAM
and advanced driving lose 0.6 to 1.4 points, within the noise; and, less expectedly, crash
mitigation (1.3 points) and emergency trajectories (1.4 points), which rank above platooning. Those
two use DS, so they have no announced reservation for pre-emption to protect, and they suffer
from the extra reselections it causes. The mean over all flows does not move. Together,
re-evaluation and pre-emption lengthen the disconnection tail of platooning (DT p99 from about 510
to 750 ms at low background), while pre-emption lowers its mean AoI at high background (81 to
68 ms). Neither mechanism changes the hidden-node collisions, 1 to 1.4 % of the failures within
300 m on both highways.

**Urban.** Buildings dominate: for every flow, 92 to 99 % of the failures within 300 m are
propagation or signals below sensitivity. Collisions are marginal, and the mechanisms change
almost nothing.

**SB-SPS against DS (CPM).** DS lowers CPM delivery against SB-SPS by 15 to 18 points within
300 m (a blind selection for every packet, nothing announced for the others to avoid), and 20 to
22 points with re-evaluation, which helps SB-SPS but not DS. Reserving for the median CPM size is
not testable with two subchannels.

**Superseded CPMs.** About 19 % of the CPMs are replaced in the queue by the next one before they
are sent: a CPM can wait up to its 100 ms PDB for its resource while the next CPM comes 50 ms +
Exp(50 ms) later. They are left out of every metric; counted as losses, CPM delivery would be 10 to
17 points lower and none of the conclusions above would change.

**Limits.** 3 seeds give wide intervals: small differences are not resolved. Retransmissions are
blind (no HARQ feedback). The SL priorities are an assumption (the PQI table gives none). The SCI
announces only the next copy of a packet. There is no run without background traffic. Emergency trajectories exist only at high
background and come from 1 to 5 vehicles per seed. Closed-loop platoon control is not modelled.